# Raster awareness API

This notebook will give an overview of newly developed raster interface. We'll cover 
basic usage of the functionality offered by the interface which mainly involves:
1. converting `xarray.DataArray` object to the PySAL's weights object (`libpysal.weights.W`/`WSP`).
2. going back to the `xarray.DataArray` from weights object.

using different datasets:
- with missing values.
- with multiple layers.
- with non conventional dimension names.

In [ ]:
import numpy as np
import requests
import xarray as xr
from esda import Moran_Local

from libpysal.weights import Queen, Rook, raster

## Loading Data

*The interface only accepts `xarray.DataArray`*, this can be easily obtained from raster data
format using `xarray`'s I/O functionality which can read from a variety of data formats some of them are listed below: 
- [GDAL Raster Formats](https://svn.osgeo.org/gdal/tags/gdal_1_2_5/frmts/formats_list.html) via `open_rasterio` method.
- [NetCDF](https://www.unidata.ucar.edu/software/netcdf/) via `open_dataset` method.

In this notebook we'll work with `NetCDF` and `GeoTIFF` data. 

### Using xarray example dataset
First lets load up a `netCDF` dataset offered by xarray.

In [ ]:
# -> returns a xarray.Dataset object
ds = xr.tutorial.open_dataset("air_temperature.nc")
da = ds["air"]  # we'll use the "air" data variable for further analysis
print(da)

`xarray`'s data structures like `Dataset` and `DataArray` provides `pandas` like functionality for multidimensional-array or ndarray. 

In our case we'll mainly deal with `DataArray`, we can see above that the `da` holds the data for air temperature, it has 2 dims coordinate dimensions `x` and `y`, and it's layered on `time` dimension so in total 3 dims (`time`, `lat`, `lon`).

We'll now group `da` by month and take average over the `time` dimension



In [ ]:
da = da.groupby("time.month").mean()
print(da.coords)  # as a result time dim is replaced by month

In [ ]:
# let's plot over month, each facet will represent
# the mean air temperature in a given month.
da.plot(
    col="month",
    col_wrap=4,
)

We can use `from_xarray` method from the contiguity classes like `Rook` and `Queen`, and also from `KNN`.

This uses a util function in `raster.py` file called `da2W`, which can also be called directly to build `W` object, similarly `da2WSP` for building `WSP` object.

**Weight builders (`from_xarray`, `da2W`, `da2WSP`) can recognise dimensions belonging to this list `[band, time, lat, y, lon, x]`, if any of the dimension in the `DataArray` does not belong to the mentioned list then we need to pass a dictionary (specifying that dimension's name) to the weight builder.** 

e.g. `dims` dictionary:
```python
>>> da.dims                  # none of the dimension belong to the default dimension list
('year', 'height', 'width')
>>> coords_labels = {                 # dimension values should be properly aligned with the following keys
        "z_label": "year",
        "y_label": "height",
        "x_label": "width"
    }
```


In [ ]:
coords_labels = {}
# since month does not belong to the default list we
# need to pass it using a dictionary
coords_labels["z_label"] = "month"
w_queen = Queen.from_xarray(
    da, z_value=12, coords_labels=coords_labels, sparse=False
)  # We'll use data from 12th layer (in our case layer=month)

`index` is a newly added attribute to the weights object, this holds the multi-indices of the non-missing values belonging to `pandas.Series` created from the passed `DataArray`, this series can be easily obtained using `DataArray.to_series()` method.

In [ ]:
w_queen.index[:5]  # indices are aligned to the ids of the weight object

We can then obtain raster data by converting the `DataArray` to `Series` and then using indices from `index` attribute to get non-missing values by subsetting the `Series`. 

In [ ]:
data = da.to_series()[w_queen.index]

We now have the required data for further analysis (we can now use methods such as ESDA/spatial regression), for this example let's compute a local Moran statistic for the extracted data.

In [ ]:
# Quickly computing and loading a LISA
np.random.seed(12345)
lisa = Moran_Local(np.array(data, dtype=np.float64), w_queen)

After getting our calculated results it's time to store them back to the `DataArray`, we can use `w2da` function directly to convert the `W` object back to `DataArray`. 

*Your use case might differ but the steps for using the interface will be similar to this example.* 

In [ ]:
# Converting obtained data back to DataArray
moran_da = raster.w2da(
    lisa.p_sim, w_queen
)  # w2da accepts list/1d array/pd.Series and a weight object aligned to passed data
print(moran_da)

In [ ]:
moran_da.plot()

### Using local `NetCDF` dataset

In the earlier example we used an example dataset from xarray for building weights object. Additonally, we had to pass the custom layer name to the builder. 

In this small example we'll build `KNN` distance weight object using a local `NetCDF` dataset with different dimensions names which doesn't belong to the default list of dimensions.

We'll also see how to speed up the reverse journey (from weights object to `DataArray`) by passing prebuilt `coords` and `attrs` to `w2da` method. 

In [ ]:
r = requests.get(
    "https://archive.unidata.ucar.edu/software/netcdf/examples/ECMWF_ERA-40_subset.nc"
)
with open("ECMWF_ERA-40_subset.nc", "wb") as f:
    f.write(r.content)

In [ ]:
# Lets load a netCDF Surface dataset
ds = xr.open_dataset(
    "ECMWF_ERA-40_subset.nc"
)  # After loading netCDF dataset we obtained a xarray.Dataset object
print(ds)  # This Dataset object containes several data variables

Out of 17 data variables we'll use `p2t` for our analysis. This will give us our desired `DataArray` object `da`, we will further group `da` by day, taking average over the `time` dimension.

In [ ]:
# this will give us the required DataArray with p2t
# (2 metre temperature) data variable
da = ds["p2t"]
da = da.groupby("time.day").mean()
print(da.dims)

**We can see that the none of dimensions of `da` matches with the default dimensions (`[band, time, lat, y, lon, x]`)**

This means we have to create a dictionary mentioning the dimensions and ship it to weight builder, similar to our last example. 

In [ ]:
coords_labels = {}
coords_labels["y_label"] = "latitude"
coords_labels["x_label"] = "longitude"
coords_labels["z_label"] = "day"
w_rook = Rook.from_xarray(da, z_value=13, coords_labels=coords_labels, sparse=True)

In [ ]:
data = da.to_series()[
    w_rook.index
]  # we derived the data from DataArray similar to our last example

In the last example we only passed the `data` values and weight object to `w2da` method, which then created the necessary `coords` to build our required `DataArray`. This process can be speed up by passing `coords` from the existing `DataArray` `da` which we used earlier.

Along with `coords` we can also pass `attrs` of the same `DataArray` this will help `w2da` to retain all the properties of original `DataArray`.

Let's compare the `DataArray` returned by `w2da` and original `DataArray`. For this we'll ship the derived data straight to `w2da` without any statistical analysis.

In [ ]:
da1 = raster.wsp2da(data, w_rook, attrs=da.attrs, coords=da[12:13].coords)
xr.DataArray.equals(
    da[12:13], da1
)  # method to compare 2 DataArray, if true then w2da was successfull

## Additional resources

1. [Reading and writing files using Xarray](http://xarray.pydata.org/en/stable/io.html)
2. [Xarray Data Structures](http://xarray.pydata.org/en/stable/data-structures.html)
3. Dataset links:
    - [ECMWF_ERA-40_subset.nc](https://www.unidata.ucar.edu/software/netcdf/examples/files.html)